In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# 작업 결과 가져오기 및 저장하기

<details>
<summary><b>패키지 버전</b></summary>

이 페이지의 코드는 다음 요구 사항을 사용하여 개발되었습니다.
이 버전 이상을 사용하시기 바랍니다.

```
qiskit-ibm-runtime~=0.40.1
```
</details>

양자 워크플로우는 완료하는 데 시간이 걸리는 경우가 많으며 여러 세션에 걸쳐 실행될 수 있습니다. Python 커널을 재시작하면 메모리에 저장된 결과가 모두 손실됩니다. 데이터 손실을 방지하기 위해 결과를 파일에 저장하고, IBM Quantum&reg;에서 이전 작업의 결과를 가져올 수 있어 다음 세션에서 이전 작업을 이어서 진행할 수 있습니다.

## IBM Quantum에서 작업 결과 가져오기
IBM Quantum은 나중에 가져올 수 있도록 모든 작업의 결과를 자동으로 저장합니다. 이 기능을 사용하여 커널 재시작 간에도 양자 프로그램을 계속 실행하고 이전 결과를 검토할 수 있습니다. 작업의 ID는 `job_id` 메서드를 통해 프로그래밍 방식으로 확인하거나, [워크로드 페이지](https://quantum.cloud.ibm.com/workloads)에서 제출한 모든 작업과 해당 ID를 확인할 수 있습니다.

작업을 프로그래밍 방식으로 찾으려면 [`QiskitRuntimeService.jobs`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/qiskit-runtime-service#jobs) 메서드를 사용하세요. 기본적으로 가장 최근에 제출된 작업이 반환되지만, Backend 이름, 생성 날짜 등으로 작업을 필터링할 수도 있습니다. 다음 셀은 지난 3개월 이내에 제출된 작업을 찾습니다. `created_after` 인수는 [`datetime.datetime`](https://docs.python.org/3.8/library/datetime.html#datetime.datetime) 객체여야 합니다.

In [1]:
import datetime
from qiskit_ibm_runtime import QiskitRuntimeService

three_months_ago = datetime.datetime.now() - datetime.timedelta(days=90)

service = QiskitRuntimeService()
jobs_in_last_three_months = service.jobs(created_after=three_months_ago)
jobs_in_last_three_months[:3]  # show first three jobs

[<RuntimeJobV2('d2n1d2cg59ks73c6vu3g', 'sampler')>,
 <RuntimeJobV2('d2n18dehb60s73cvalg0', 'sampler')>,
 <RuntimeJobV2('d2ff2cms6qcs738f4lb0', 'sampler')>]

You can also select by backend, job state, session, and more. For more information, see [`QiskitRuntimeService.jobs`](/docs/api/qiskit-ibm-runtime/qiskit-runtime-service#jobs) in the API documentation.

Once you have the job ID, use the [`QiskitRuntimeService.job`](/docs/api/qiskit-ibm-runtime/qiskit-runtime-service#job) method to retrieve it.

In [2]:
# Get ID of most recent successful job for demonstration.
# This will not work if you've never successfully run a job.
successful_job = next(
    j for j in service.jobs(limit=1000) if j.status() == "DONE"
)
job_id = successful_job.job_id()
print(job_id)

d2n1d2cg59ks73c6vu3g


In [3]:
retrieved_job = service.job(job_id)
retrieved_job.result()

PrimitiveResult([SamplerPubResult(data=DataBin(c=BitArray(<shape=(), num_shots=4096, num_bits=2>)), metadata={'circuit_metadata': {}}), SamplerPubResult(data=DataBin(c=BitArray(<shape=(), num_shots=4096, num_bits=2>)), metadata={'circuit_metadata': {}}), SamplerPubResult(data=DataBin(c=BitArray(<shape=(), num_shots=4096, num_bits=2>)), metadata={'circuit_metadata': {}}), SamplerPubResult(data=DataBin(c=BitArray(<shape=(), num_shots=4096, num_bits=2>)), metadata={'circuit_metadata': {}}), SamplerPubResult(data=DataBin(c=BitArray(<shape=(), num_shots=4096, num_bits=2>)), metadata={'circuit_metadata': {}}), SamplerPubResult(data=DataBin(c=BitArray(<shape=(), num_shots=4096, num_bits=2>)), metadata={'circuit_metadata': {}}), SamplerPubResult(data=DataBin(c=BitArray(<shape=(), num_shots=4096, num_bits=2>)), metadata={'circuit_metadata': {}}), SamplerPubResult(data=DataBin(c=BitArray(<shape=(), num_shots=4096, num_bits=2>)), metadata={'circuit_metadata': {}}), SamplerPubResult(data=DataBin(c

<CodeAssistantAdmonition
  tagLine="Always forgetting how to retrieve a job result? Try this prompt with Qiskit Code Assistant:"
  prompts={[
    "# Retrieve job JOB_ID from IBM Runtime and print the result"
  ]}
/>

## Save results to disk

You may also want to save results to disk. To do this, use Python's built-in JSON library with encoders from Qiskit Runtime.

In [4]:
import json
from qiskit_ibm_runtime import RuntimeEncoder

with open("result.json", "w") as file:
    json.dump(retrieved_job.result(), file, cls=RuntimeEncoder)

You can then load this array from disk in a separate kernel.

In [5]:
from qiskit_ibm_runtime import RuntimeDecoder

with open("result.json", "r") as file:
    result = json.load(file, cls=RuntimeDecoder)

result

PrimitiveResult([SamplerPubResult(data=DataBin(c=BitArray(<shape=(), num_shots=4096, num_bits=2>)), metadata={'circuit_metadata': {}}), SamplerPubResult(data=DataBin(c=BitArray(<shape=(), num_shots=4096, num_bits=2>)), metadata={'circuit_metadata': {}}), SamplerPubResult(data=DataBin(c=BitArray(<shape=(), num_shots=4096, num_bits=2>)), metadata={'circuit_metadata': {}}), SamplerPubResult(data=DataBin(c=BitArray(<shape=(), num_shots=4096, num_bits=2>)), metadata={'circuit_metadata': {}}), SamplerPubResult(data=DataBin(c=BitArray(<shape=(), num_shots=4096, num_bits=2>)), metadata={'circuit_metadata': {}}), SamplerPubResult(data=DataBin(c=BitArray(<shape=(), num_shots=4096, num_bits=2>)), metadata={'circuit_metadata': {}}), SamplerPubResult(data=DataBin(c=BitArray(<shape=(), num_shots=4096, num_bits=2>)), metadata={'circuit_metadata': {}}), SamplerPubResult(data=DataBin(c=BitArray(<shape=(), num_shots=4096, num_bits=2>)), metadata={'circuit_metadata': {}}), SamplerPubResult(data=DataBin(c